Loads the dataset from CSV into a DataFrame and previews the first rows.

In [ ]:
import pandas as pd
import numpy as np

# Load the cleaned and corrected dataset (solarpath_day1_clean_dataset.csv)
# This replaces the old sequence_dataset_test.csv which was removed.
# SOURCE_KEY is present in this file and will be retained for per-inverter anomaly grouping.
df = pd.read_csv('solarpath_day1_clean_dataset.csv')
df.head()

In [ ]:
df.value_counts('site_id').sum()

Makes a working copy of the data and parses/sorts the timestamp column so the sequence is in chronological order.

In [ ]:
# Working copy of the raw data
df_feat = df.copy()

# Parse timestamps and sort chronologically (handles mixed date formats)
if 'timestamp' in df_feat.columns:
    df_feat['timestamp'] = pd.to_datetime(
        df_feat['timestamp'], format='mixed', errors='coerce'
    )
    df_feat = df_feat.sort_values('timestamp').reset_index(drop=True)

Builds lag and rolling-window features (previous values, rolling mean/std) from the weather-related columns.

In [ ]:
# Weather lag features (previous readings + rolling stats)
# Note: soil_wetness removed — not available in solarpath_day1_clean_dataset.csv
weather_cols = [
    'temp_module',
    'temp_ambient',
    'humidity',
    'wind_speed',
]

for col in weather_cols:
    if col in df_feat.columns:
        df_feat[f'{col}_lag1'] = df_feat[col].shift(1)
        df_feat[f'{col}_lag2'] = df_feat[col].shift(2)
        df_feat[f'{col}_roll_mean_3'] = (
            df_feat[col].rolling(window=3, min_periods=1).mean()
        )
        df_feat[f'{col}_roll_std_3'] = (
            df_feat[col].rolling(window=3, min_periods=1).std().fillna(0)
        )

Adds a module-vs-ambient temperature delta feature, drops rows with missing values, and defines the final feature list.

In [ ]:
# Thermal delta between module and ambient temperature
if 'temp_module' in df_feat.columns and 'temp_ambient' in df_feat.columns:
    df_feat['temp_delta'] = df_feat['temp_module'] - df_feat['temp_ambient']

# Drop rows with NaNs introduced by the lag/shift operations
df_feat = df_feat.dropna().reset_index(drop=True)

# Columns that shouldn't be used as model features
# actual_dc_power, actual_ac_power, latitude, longitude are extra in solarpath_day1_clean_dataset.csv
# SOURCE_KEY is kept in df_feat for anomaly grouping but excluded from model features
drop_cols = [
    'timestampsite_id',
    'timestamp',
    'site_id',
    'SOURCE_KEY',
    'is_daylight',
    'irradiation',
    'actual_dc_power',
    'actual_ac_power',
    'latitude',
    'longitude',
]
features = [
    c for c in df_feat.columns if c not in drop_cols and c != 'actual_ratio'
]

Splits the features and target into train/validation/test sets in chronological order and prints the feature count.

In [ ]:
X = df_feat[features]
y = df_feat['actual_ratio']

# Time-series split (no shuffling, since order matters)
total = len(df_feat)
train_end = int(total * 0.70)
val_end = int(total * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

print(f"Generated {len(features)} features:")
print(features)

Imports the regression models and metrics, and defines the individual base models to compare.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Individual base models used for comparison
models = {
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, max_depth=12, random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBRegressor(
        n_estimators=150,
        max_depth=6,
        learning_rate=0.03,
        random_state=42,
        n_jobs=-1,
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42
    ),
}

results = []

Trains each base model, computes validation/test metrics, and builds a benchmark table sorted by test R2.

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    val_r2, val_mae = r2_score(y_val, val_pred), mean_absolute_error(y_val, val_pred)
    test_r2, test_mae = r2_score(y_test, test_pred), mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

    results.append({
        'Model': name,
        'Val R2': val_r2,
        'Test R2': test_r2,
        'Test MAE': test_mae,
        'Test RMSE': test_rmse,
    })
    print(f"{name} trained.")

benchmark_df = pd.DataFrame(results).sort_values(by='Test R2', ascending=False)
print("\nIndividual models benchmark:")
display(benchmark_df)

Defines the base estimators (Random Forest, XGBoost, Gradient Boosting) used inside the stacking ensemble.

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression

# Base estimators for the stacking ensemble
estimators = [
    (
        'rf',
        RandomForestRegressor(
            n_estimators=100, max_depth=12, random_state=42, n_jobs=-1
        ),
    ),
    (
        'xgb',
        XGBRegressor(
            n_estimators=150,
            max_depth=6,
            learning_rate=0.03,
            random_state=42,
            n_jobs=-1,
        ),
    ),
    (
        'gb',
        GradientBoostingRegressor(
            n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42
        ),
    ),
]

Trains the stacking regressor on top of the base estimators and adds its results to the benchmark table.

In [ ]:
stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression(),
    cv=3,
    n_jobs=-1,
)

stacking_model.fit(X_train, y_train)

val_preds_stack = stacking_model.predict(X_val)
test_preds_stack = stacking_model.predict(X_test)

stacking_metrics = {
    'Model': 'Stacking Ensemble (Proposed)',
    'Val R2': r2_score(y_val, val_preds_stack),
    'Test R2': r2_score(y_test, test_preds_stack),
    'Test MAE': mean_absolute_error(y_test, test_preds_stack),
    'Test RMSE': np.sqrt(mean_squared_error(y_test, test_preds_stack)),
}

full_benchmark = pd.concat(
    [benchmark_df, pd.DataFrame([stacking_metrics])], ignore_index=True
)
print("Final model comparison:")
display(
    full_benchmark.sort_values(by='Test R2', ascending=False).reset_index(drop=True)
)

**Plots actual vs. predicted performance ratio over a smoothed 96-sample window, with predictions clipped to zero during nighttime to match the real-world daylight-only generation pattern.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Smooth predictions with a rolling average
preds_smoothed = pd.Series(test_preds_stack).rolling(window=3, min_periods=1).mean().values

# Enforce zero output when actual is strictly 0 (nighttime baseline alignment)
preds_cleaned = np.where(y_test.values == 0, 0.0, preds_smoothed)

# Slice a 96-sample window for a clearer view
sample_start_idx = 475
num_samples = 96
sample_end_idx = min(sample_start_idx + num_samples, len(y_test))

y_test_slice = y_test.iloc[sample_start_idx:sample_end_idx].values
preds_slice = preds_cleaned[sample_start_idx:sample_end_idx]

plt.figure(figsize=(12, 5))
plt.plot(
    y_test_slice,
    label='Actual Performance Ratio',
    color='black',
    linewidth=2,
    alpha=0.85,
)
plt.plot(
    preds_slice,
    label='Predicted (Stacking Model - Cleaned)',
    color='#1f77b4',
    linestyle='--',
    linewidth=2,
)

plt.title('Actual vs Predicted Solar Performance Ratio (Cleaned 96-Sample Window)', fontsize=12)
plt.xlabel('Sample Index (Time Step)')
plt.ylabel('Performance Ratio (PR)')
plt.ylim(-0.05, 1.1)
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

**Compares the test R2 score of all trained models, including the stacking ensemble, in a single bar chart.**

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.barplot(data=full_benchmark, x='Test R2', y='Model', palette='Blues_r')
plt.title('Model Comparison - Test R2 Score')
plt.xlim(0, 1.0)
plt.tight_layout()
plt.show()

**Ranks the top 15 features by importance from the XGBoost base model to confirm the model relies on physically meaningful signals.**

In [ ]:
xgb_base = stacking_model.named_estimators_['xgb']

importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': xgb_base.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'][:15][::-1], importance_df['Importance'][:15][::-1])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances (XGBoost Base Model)')
plt.tight_layout()
plt.show()

**Applying the trained model to estimate power output for an Egypt-based site, using the predicted performance ratio combined with assumed site area and panel efficiency. This formula and its assumptions are documented as a study limitation.**

In [ ]:
Area_Egypt = 1600 * 600      # m^2, real 50 MW sub-plant plot size at Benban
Efficiency_Egypt = 0.17      # ~17%, based on real 330W/1.94m^2 module specs

predicted_power_egypt = test_preds_stack * Area_Egypt * Efficiency_Egypt

power_df = pd.DataFrame({
    'Predicted_Performance_Ratio': test_preds_stack,
    'Predicted_Power_Egypt_kW': predicted_power_egypt
})

power_df.head(10)

In [ ]:
power_df.to_csv('power_df.csv')

In [ ]:
import joblib

joblib.dump(stacking_model, 'stacking_solar_model.pkl')


Anomaly Detection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Combine validation and test predictions
eval_actual = pd.concat([y_val, y_test]).reset_index(drop=True)
eval_pred = np.concatenate([val_preds_stack, test_preds_stack])

# Same rows corresponding to validation + test
anomaly_df = df_feat.iloc[train_end:].copy().reset_index(drop=True)

# Actual and expected performance ratios
anomaly_df['Actual_Ratio'] = eval_actual
anomaly_df['Expected_Ratio'] = eval_pred

# Match the existing notebook behavior:
# generation is zero during nighttime
anomaly_df.loc[~anomaly_df['is_daylight'], 'Expected_Ratio'] = 0

# Main anomaly feature
anomaly_df['Deviation'] = (
    anomaly_df['Actual_Ratio'] - anomaly_df['Expected_Ratio']
)

print("Anomaly dataset shape:", anomaly_df.shape)
anomaly_df[
    [
        'timestamp',
        'irradiation',
        'temp_ambient',
        'temp_module',
        'temp_delta',
        'Actual_Ratio',
        'Expected_Ratio',
        'Deviation',
        'is_daylight'
    ]
].head()

In [ ]:
anomaly_df[
    [
        'Actual_Ratio',
        'Expected_Ratio',
        'Deviation'
    ]
].describe()

In [ ]:
daylight_df = anomaly_df[
    anomaly_df['is_daylight'] == True
].copy()

print("Daylight readings:", len(daylight_df))

In [ ]:
anomaly_features = [
    'Actual_Ratio',
    'Expected_Ratio',
    'Deviation',
    'irradiation',
    'temp_delta'
]

X_anomaly = daylight_df[anomaly_features].dropna()

print("Features used for anomaly detection:")
print(anomaly_features)

print("\nShape:", X_anomaly.shape)

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    daylight_df['Deviation'].dropna(),
    bins=50
)

plt.axvline(
    0,
    linestyle='--',
    label='Zero Deviation'
)

plt.xlabel('Deviation')
plt.ylabel('Frequency')
plt.title('Distribution of Deviation')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    daylight_df['timestamp'],
    daylight_df['Deviation'],
    linewidth=0.7
)

plt.axhline(
    0,
    linestyle='--'
)

plt.xlabel('Time')
plt.ylabel('Deviation')
plt.title('Deviation Over Time')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
valid_daylight_df = daylight_df[
    ~(
        (daylight_df['Actual_Ratio'] == 0) &
        (daylight_df['irradiation'] > 0)
    )
].copy()

print("Original daylight rows:", len(daylight_df))
print("Valid daylight rows:", len(valid_daylight_df))
print(
    "Removed rows:",
    len(daylight_df) - len(valid_daylight_df)
)

In [ ]:
anomaly_features = [
    'Actual_Ratio',
    'Expected_Ratio',
    'Deviation',
    'irradiation',
    'temp_delta'
]

X_anomaly = valid_daylight_df[anomaly_features].dropna()

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

scaler_if = StandardScaler()
X_scaled = scaler_if.fit_transform(X_anomaly)

iso_forest = IsolationForest(
    n_estimators=200,
    contamination='auto',
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(X_scaled)

if_prediction = iso_forest.predict(X_scaled)
if_score = -iso_forest.decision_function(X_scaled)

daylight_result = valid_daylight_df.loc[
    X_anomaly.index
].copy()

daylight_result['IF_Label'] = if_prediction
daylight_result['IF_Anomaly_Score'] = if_score

daylight_result['IF_Status'] = daylight_result[
    'IF_Label'
].map({
    1: 'Normal',
    -1: 'Anomaly'
})

print(daylight_result['IF_Status'].value_counts())

In [ ]:
print(
    daylight_result['IF_Status']
    .value_counts(normalize=True)
    .mul(100)
)

print("\nAnomaly Deviation Statistics:")
print(
    daylight_result[
        daylight_result['IF_Status'] == 'Anomaly'
    ]['Deviation'].describe()
)


In [ ]:
detected_anomalies = daylight_result[
    daylight_result['IF_Status'] == 'Anomaly'
].copy()

negative_anomalies = detected_anomalies[
    detected_anomalies['Deviation'] < 0
].copy()

print("Total anomalies:", len(detected_anomalies))
print("Negative-deviation anomalies:", len(negative_anomalies))

print(
    f"Negative-deviation anomaly rate: "
    f"{len(negative_anomalies) / len(detected_anomalies) * 100:.2f}%"
)

In [ ]:
critical_anomalies = negative_anomalies.sort_values(
    'Deviation'
).head(20)

critical_anomalies[
    [
        'timestamp',
        'Actual_Ratio',
        'Expected_Ratio',
        'Deviation',
        'irradiation',
        'temp_ambient',
        'temp_module',
        'temp_delta',
        'IF_Anomaly_Score'
    ]
]

In [ ]:
critical_anomalies = (
    detected_anomalies
    .sort_values('Deviation')
    .head(20)
)

critical_anomalies[
    [
        'timestamp',
        'Actual_Ratio',
        'Expected_Ratio',
        'Deviation',
        'irradiation',
        'temp_ambient',
        'temp_module',
        'temp_delta',
        'IF_Anomaly_Score'
    ]
]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.scatter(
    daylight_result['timestamp'],
    daylight_result['Deviation'],
    s=8,
    label='All readings'
)

plt.scatter(
    critical_anomalies['timestamp'],
    critical_anomalies['Deviation'],
    s=35,
    label='Critical negative-deviation anomalies'
)

plt.axhline(0, linestyle='--')

plt.xlabel('Time')
plt.ylabel('Deviation')
plt.title('Detected Anomalies Based on Negative Deviation')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

plt.scatter(
    daylight_result['timestamp'],
    daylight_result['Deviation'],
    s=8
)

plt.axhline(
    0,
    linestyle='--'
)

plt.xlabel('Time')
plt.ylabel('Deviation')
plt.title('Negative Deviation as an Indicator of Potential Faults')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
try:
    from tensorflow.keras.models import Model
    from tensorflow.keras.layers import Input, Dense

    print("TensorFlow is available.")

except ImportError:
    print("TensorFlow is not installed.")

In [ ]:
input_dim = X_scaled.shape[1]

input_layer = Input(shape=(input_dim,))

encoded = Dense(
    16,
    activation='relu'
)(input_layer)

encoded = Dense(
    8,
    activation='relu'
)(encoded)

decoded = Dense(
    16,
    activation='relu'
)(encoded)

decoded = Dense(
    input_dim,
    activation='linear'
)(decoded)

autoencoder = Model(
    input_layer,
    decoded
)

autoencoder.compile(
    optimizer='adam',
    loss='mse'
)

autoencoder.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='loss',
    patience=5,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_scaled,
    X_scaled,
    epochs=50,
    batch_size=256,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
reconstructed = autoencoder.predict(
    X_scaled,
    verbose=0
)

reconstruction_error = np.mean(
    np.square(X_scaled - reconstructed),
    axis=1
)

daylight_result['AE_Error'] = reconstruction_error

threshold = np.percentile(
    reconstruction_error,
    95
)

daylight_result['AE_Status'] = np.where(
    reconstruction_error > threshold,
    'Anomaly',
    'Normal'
)

print("Autoencoder threshold:", threshold)

print("\nAutoencoder results:")
print(daylight_result['AE_Status'].value_counts())

In [ ]:
comparison = daylight_result[
    [
        'timestamp',
        'Actual_Ratio',
        'Expected_Ratio',
        'Deviation',
        'IF_Status',
        'IF_Anomaly_Score',
        'AE_Status',
        'AE_Error'
    ]
].copy()

print("Isolation Forest:")
print(comparison['IF_Status'].value_counts())
print()

print("Autoencoder:")
print(comparison['AE_Status'].value_counts())

In [ ]:
if_anomalies = daylight_result[
    daylight_result['IF_Status'] == 'Anomaly'
]

ae_anomalies = daylight_result[
    daylight_result['AE_Status'] == 'Anomaly'
]

if_negative = if_anomalies[
    if_anomalies['Deviation'] < 0
]

ae_negative = ae_anomalies[
    ae_anomalies['Deviation'] < 0
]

print("Isolation Forest")
print("Total anomalies:", len(if_anomalies))
print("Negative-deviation anomalies:", len(if_negative))
print(
    "Negative-deviation rate:",
    f"{len(if_negative) / len(if_anomalies) * 100:.2f}%"
)

print("\nAutoencoder")
print("Total anomalies:", len(ae_anomalies))
print("Negative-deviation anomalies:", len(ae_negative))
print(
    "Negative-deviation rate:",
    f"{len(ae_negative) / len(ae_anomalies) * 100:.2f}%"
)

In [ ]:
final_ae_cases = daylight_result[
    (daylight_result['AE_Status'] == 'Anomaly') &
    (daylight_result['Deviation'] < 0)
].sort_values('Deviation')

print("Potential anomalies detected by Autoencoder:")
print("Count:", len(final_ae_cases))

final_ae_cases[
    [
        'timestamp',
        'Actual_Ratio',
        'Expected_Ratio',
        'Deviation',
        'irradiation',
        'temp_ambient',
        'temp_module',
        'temp_delta',
        'AE_Error'
    ]
].head(20)

In [ ]:
print("""
ANOMALY DETECTION PIPELINE
==========================

1. Actual_Ratio:
   The actual performance ratio available in the dataset.

2. Expected_Ratio:
   The expected performance ratio estimated by the Stacking Regression model.

3. Deviation:
   Deviation = Actual_Ratio - Expected_Ratio

4. Anomaly Features:
   - Actual_Ratio
   - Expected_Ratio
   - Deviation
   - irradiation
   - temp_delta

5. Isolation Forest:
   Detects unusual observations based on the feature space.

6. Autoencoder:
   Detects unusual observations using reconstruction error.

7. Main fault indicator:
   Large negative Deviation means that the actual performance
   is much lower than the expected performance.

Note:
The current dataset does not contain true fault labels.
Therefore, detected anomalies are treated as potential anomalies
or potential performance problems, not confirmed physical faults.
""")

In [ ]:
print("""
WHY RATIO-BASED FEATURES ARE USED
=================================

Ratio-based features describe performance relative to expected
performance rather than relying only on absolute power values.

Using relative performance makes the anomaly-detection pipeline
more transferable across solar plants with different capacities,
numbers of panels, and operating scales.

The same anomaly-detection architecture can therefore be reused
when Egyptian operational data becomes available.

For the current implementation:
Actual_Ratio and Expected_Ratio are used to calculate Deviation,
and Deviation is the main indicator of under-performance.
""")

In [ ]:
print("""
INDIA DATASET -> FUTURE EGYPTIAN DATA
=====================================

Current pipeline:
Weather / operational data
        ↓
Feature engineering
        ↓
Stacking Regression
        ↓
Expected performance ratio
        ↓
Deviation
        ↓
Isolation Forest / Autoencoder
        ↓
Anomaly detection

Future Egyptian data:
Replace the current operational/weather observations
with real Egyptian solar-plant observations.

The anomaly-detection architecture remains the same:
- same feature concept
- same Deviation calculation
- same Isolation Forest structure
- same Autoencoder structure

Only the input data and model calibration may need to be
validated or adjusted for the characteristics of the Egyptian plant.

No complete redesign of the anomaly-detection architecture
is required.
""")

## Conclusion

The anomaly-detection stage uses the predicted performance ratio from
the Stacking Regression model together with the observed `actual_ratio`
to calculate `Deviation`. The anomaly-detection feature set consists of
`Actual_Ratio`, `Expected_Ratio`, `Deviation`, `irradiation`, and
`temp_delta`.

Isolation Forest identified 3,376 anomalous readings, of which 920
(27.25%) had negative deviation. The Autoencoder identified 1,024
anomalous readings, of which 394 (38.48%) had negative deviation.

Large negative deviation is treated as the strongest indicator of
potential under-performance because it means that observed performance
is substantially lower than the model-expected performance.

The current dataset does not contain confirmed fault labels, so the
detected cases should be interpreted as potential anomalies or
performance issues rather than confirmed physical faults.

The relative-ratio formulation also supports future reuse with Egyptian
solar-plant data. Once real Egyptian operational and weather data are
available, the same anomaly-detection architecture can be applied,
with validation and possible calibration adjustments for the specific
plant.

In [ ]:
daylight_df[
    (daylight_df['Actual_Ratio'] == 0) &
    (daylight_df['irradiation'] > 0)
][
    [
        'timestamp',
        'irradiation',
        'Actual_Ratio',
        'Expected_Ratio',
        'Deviation',
        'is_daylight'
    ]
].head(20)

In [ ]:
valid_daylight_df = daylight_df[
    ~(
        (daylight_df['Actual_Ratio'] == 0) &
        (daylight_df['irradiation'] > 0)
    )
].copy()

print("Original daylight rows:", len(daylight_df))
print("Valid daylight rows:", len(valid_daylight_df))
print(
    "Removed rows:",
    len(daylight_df) - len(valid_daylight_df)
)

In [ ]:
import joblib

valid_daylight_df.to_csv('valid_daylight_df.csv', index=False)
joblib.dump(valid_daylight_df, 'valid_daylight_df.joblib')

print("Data saved successfully:")
print("- valid_daylight_df.csv")
print("- valid_daylight_df.joblib")